In [113]:
import pandas as pd
import numpy as np

In [114]:
products = pd.read_csv('/content/combined_products_2025_v2-2.csv')

In [115]:
suppliers = pd.read_csv('/content/synthetic_suppliers.csv')

In [116]:
products.loc[lambda x: x['product'] == 'power_devices']

,date,ppi_value,country_code,ppi_source,real_price,product,ppi_region,year,month
22700,2012-02-01,NaN,JPN,FRED,NaN,power_devices,OAC,2012,2
22701,2012-02-01,83.8000,DEU,FRED,0.125700,power_devices,USA,2012,2
22702,2012-02-01,83.8000,FRA,FRED,0.117320,power_devices,USA,2012,2
22703,2012-02-01,83.8000,GBR,FRED,0.117320,power_devices,USA,2012,2
22704,2012-02-01,83.8000,NLD,FRED,0.117320,power_devices,USA,2012,2
...,...,...,...,...,...,...,...,...,...
24365,2025-12-01,98.7000,SGP,FRED,0.088830,power_devices,OAC,2025,12
24366,2025-12-01,98.7000,MYS,FRED,0.078960,power_devices,OAC,2025,12
24367,2025-12-01,98.7000,THA,FRED,0.076986,power_devices,OAC,2025,12
24368,2025-12-01,88.2378,CHN,FRED,0.061766,power_devices,CHN,2025,12


In [117]:
suppliers

,country_code,supplier_id,lead_time_mean,lead_time_variance,disruption_probability,compliance_eligibility,logistics_reliability
0,ARE,SUP_ARE_1,29.081672,2.427785,0.283742,0.728946,0.844884
1,AUS,SUP_AUS_2,53.948194,2.954602,0.200381,0.789128,0.779876
2,AUS,SUP_AUS_3,53.655471,8.445529,0.226886,0.820285,0.768079
3,BEL,SUP_BEL_4,30.632043,5.234186,0.274002,0.725275,0.795612
4,BEL,SUP_BEL_5,33.266109,2.126099,0.242300,0.801282,0.832547
...,...,...,...,...,...,...,...
86,THA,SUP_THA_87,36.172719,4.173442,0.415005,0.475663,0.719665
87,THA,SUP_THA_88,36.522238,4.386632,0.424040,0.503805,0.673557
88,THA,SUP_THA_89,39.347376,2.127328,0.471242,0.455950,0.704292
89,USA,SUP_USA_90,11.424375,1.132189,0.315940,0.713011,0.808574


In [118]:
suppliers.country_code.unique()

array(['ARE', 'AUS', 'BEL', 'BRA', 'CAN', 'CHN', 'DEU', 'FIN', 'FRA',
       'GBR', 'HKG', 'IDN', 'IND', 'JPN', 'MEX', 'MYS', 'NLD', 'SGP',
       'THA', 'USA'], dtype=object)

In [119]:
products['product'].unique()

array(['integrated_circuit_components', 'microprocessors',
       'power_devices', 'transistors'], dtype=object)

# Attaching Products Randomly to Suppliers by Weighted-Tier Levels

In [120]:
# ============================================================
# Country-Specific, Industry-Aligned Product Supply Weights
# ============================================================

country_product_weights = {

    # -------------------------
    # Tier 1A — IC Component & OSAT Powerhouses
    # -------------------------
    "CHN": {
        "Microprocessors": 0.10,
        "Power Devices": 0.20,
        "Transistors": 0.25,
        "IC Components": 0.45
    },
    "MYS": {
        "Microprocessors": 0.10,
        "Power Devices": 0.20,
        "Transistors": 0.20,
        "IC Components": 0.50
    },
    "THA": {
        "Microprocessors": 0.05,
        "Power Devices": 0.25,
        "Transistors": 0.20,
        "IC Components": 0.50
    },
    "SGP": {
        "Microprocessors": 0.15,
        "Power Devices": 0.20,
        "Transistors": 0.15,
        "IC Components": 0.50
    },

    # -------------------------
    # Tier 1B — Advanced Logic / Specialty Analog
    # -------------------------
    "USA": {
        "Microprocessors": 0.60,
        "Power Devices": 0.05,
        "Transistors": 0.30,
        "IC Components": 0.05
    },
    "JPN": {
        "Microprocessors": 0.20,
        "Power Devices": 0.40,
        "Transistors": 0.25,
        "IC Components": 0.15
    },
    "DEU": {
        "Microprocessors": 0.10,
        "Power Devices": 0.45,
        "Transistors": 0.25,
        "IC Components": 0.20
    },

    # -------------------------
    # Tier 2 — Moderate Semiconductor Presence
    # -------------------------
    "MEX": {
        "Microprocessors": 0.10,
        "Power Devices": 0.25,
        "Transistors": 0.30,
        "IC Components": 0.35
    },
    "NLD": {
        "Microprocessors": 0.20,
        "Power Devices": 0.25,
        "Transistors": 0.25,
        "IC Components": 0.30
    },
    "CAN": {
        "Microprocessors": 0.20,
        "Power Devices": 0.05,
        "Transistors": 0.35,
        "IC Components": 0.40
    },
    "FRA": {
        "Microprocessors": 0.15,
        "Power Devices": 0.30,
        "Transistors": 0.25,
        "IC Components": 0.30
    },
    "GBR": {
        "Microprocessors": 0.20,
        "Power Devices": 0.20,
        "Transistors": 0.30,
        "IC Components": 0.30
    },
    "IND": {
        "Microprocessors": 0.10,
        "Power Devices": 0.25,
        "Transistors": 0.30,
        "IC Components": 0.35
    },
    "HKG": {
        "Microprocessors": 0.10,
        "Power Devices": 0.05,
        "Transistors": 0.40,
        "IC Components": 0.45
    },
    "IDN": {
        "Microprocessors": 0.10,
        "Power Devices": 0.05,
        "Transistors": 0.40,
        "IC Components": 0.45
    },

    # -------------------------
    # Tier 3 — Low Semiconductor Strength
    # -------------------------
    "ARE": {
        "Microprocessors": 0.05,
        "Power Devices": 0.15,
        "Transistors": 0.55,
        "IC Components": 0.25
    },
    "AUS": {
        "Microprocessors": 0.10,
        "Power Devices": 0.05,
        "Transistors": 0.55,
        "IC Components": 0.30
    },
    "BEL": {
        "Microprocessors": 0.10,
        "Power Devices": 0.20,
        "Transistors": 0.40,
        "IC Components": 0.30
    },
    "BRA": {
        "Microprocessors": 0.05,
        "Power Devices": 0.20,
        "Transistors": 0.45,
        "IC Components": 0.30
    },
    "FIN": {
        "Microprocessors": 0.10,
        "Power Devices": 0.20,
        "Transistors": 0.40,
        "IC Components": 0.30
    }
}


In [121]:
# ============================================================
# Function to sample a product for each supplier
# ============================================================

random_state = 42

def assign_product(country_code):
    weights = country_product_weights[country_code]
    products = list(weights.keys())
    probs = np.array(list(weights.values()))
    return np.random.choice(products, p=probs)

# ============================================================
# Apply to your suppliers DataFrame
# ============================================================

suppliers["product"] = suppliers["country_code"].apply(assign_product)


In [122]:
suppliers.tail(30)

,country_code,supplier_id,lead_time_mean,lead_time_variance,disruption_probability,compliance_eligibility,logistics_reliability,product
61,MYS,SUP_MYS_62,33.322066,3.306858,0.407854,0.653027,0.711920,Transistors
62,MYS,SUP_MYS_63,37.240200,7.342742,0.382905,0.649724,0.699997,IC Components
63,MYS,SUP_MYS_64,37.008448,7.193708,0.362320,0.550479,0.723236,Transistors
64,MYS,SUP_MYS_65,36.996900,5.214022,0.381020,0.583543,0.733576,Transistors
65,MYS,SUP_MYS_66,37.680903,1.893604,0.412508,0.601849,0.743975,IC Components
66,MYS,SUP_MYS_67,39.577049,4.141074,0.347457,0.616875,0.715639,Power Devices
67,MYS,SUP_MYS_68,38.878107,6.141308,0.364970,0.574475,0.721331,Power Devices
68,NLD,SUP_NLD_69,35.529276,2.798855,0.219486,0.818915,0.842397,Microprocessors
69,NLD,SUP_NLD_70,33.360303,3.700034,0.226659,0.806282,0.798177,Transistors
70,NLD,SUP_NLD_71,36.251491,3.564877,0.210003,0.868677,0.822919,Transistors


# Creating a Probability of Defect column based on what the country behind the supplier *tends* to manufacture in the modern semiconductor industry

Criteria:
 - How aligned the supplier's assigned product is with what the country is actually known for producing
 - how mature and reliable that country's manufacturing base is
 - How complex the product category is (microprocessors are more difficult to manufacture; thus, higher defect risk)
 - Added a little noise for variation within each countries list of suppliers

In [123]:
import numpy as np

# ------------------------------------------------------------
# Baseline difficulty per product
# ------------------------------------------------------------
product_difficulty = {
    "Microprocessors": 0.08,
    "Power Devices": 0.05,
    "Transistors": 0.03,
    "IC Components": 0.04
}

# ------------------------------------------------------------
# Function to compute probability_of_defect
# ------------------------------------------------------------

random_state = 42

def compute_defect_probability(country_code, product):
    # Get the country’s weight for this product
    weight = country_product_weights[country_code][product]

    # Alignment factor (higher = worse alignment)
    alignment_factor = 1 - weight

    # Baseline difficulty
    difficulty = product_difficulty[product]

    # Final defect probability
    defect_prob = difficulty * (0.5 + 0.5 * alignment_factor) * np.random.uniform(0.9, 1.1)

    return defect_prob


In [124]:
# ------------------------------------------------------------
# Apply to suppliers DataFrame
# ------------------------------------------------------------
suppliers["probability_of_defect"] = suppliers.apply(
    lambda row: compute_defect_probability(row["country_code"], row["product"]),
    axis=1
)


In [125]:
suppliers.tail(30)

,country_code,supplier_id,lead_time_mean,lead_time_variance,disruption_probability,compliance_eligibility,logistics_reliability,product,probability_of_defect
61,MYS,SUP_MYS_62,33.322066,3.306858,0.407854,0.653027,0.711920,Transistors,0.027868
62,MYS,SUP_MYS_63,37.240200,7.342742,0.382905,0.649724,0.699997,IC Components,0.027958
63,MYS,SUP_MYS_64,37.008448,7.193708,0.362320,0.550479,0.723236,Transistors,0.026254
64,MYS,SUP_MYS_65,36.996900,5.214022,0.381020,0.583543,0.733576,Transistors,0.027649
65,MYS,SUP_MYS_66,37.680903,1.893604,0.412508,0.601849,0.743975,IC Components,0.028246
66,MYS,SUP_MYS_67,39.577049,4.141074,0.347457,0.616875,0.715639,Power Devices,0.047457
67,MYS,SUP_MYS_68,38.878107,6.141308,0.364970,0.574475,0.721331,Power Devices,0.048676
68,NLD,SUP_NLD_69,35.529276,2.798855,0.219486,0.818915,0.842397,Microprocessors,0.078337
69,NLD,SUP_NLD_70,33.360303,3.700034,0.226659,0.806282,0.798177,Transistors,0.027282
70,NLD,SUP_NLD_71,36.251491,3.564877,0.210003,0.868677,0.822919,Transistors,0.028449


# Adding Bulk-Order Discount Rate Column

Criteria:
- how mature and high-volume a country's semiconductor manufacturing base is
- how aligned the supplier's assigned product is with what that country is actually known for producing
- how competitive the country is in packaging markets
- how aggressively suppliers in that country typically discount for volume

In [126]:
random_state = 42

# ------------------------------------------------------------
# Base bulk discount by product (industry realistic)
# ------------------------------------------------------------
base_bulk_discount = {
    "Microprocessors": 0.05,   # 5%
    "Power Devices": 0.08,     # 8%
    "Transistors": 0.12,       # 12%
    "IC Components": 0.15      # 15%
}

# ------------------------------------------------------------
# Compute bulk discount based on country-product alignment
# ------------------------------------------------------------
def compute_bulk_discount(country_code, product):
    # Country's strength in this product
    weight = country_product_weights[country_code][product]

    # Base discount for this product category
    base = base_bulk_discount[product]

    # Final discount: stronger alignment → deeper discount
    discount = base * (0.5 + 0.5 * weight) * np.random.uniform(0.9, 1.1)

    return discount

# ------------------------------------------------------------
# Apply to suppliers DataFrame
# ------------------------------------------------------------
suppliers["bulk_discount"] = suppliers.apply(
    lambda row: compute_bulk_discount(row["country_code"], row["product"]),
    axis=1
)


In [127]:
suppliers

,country_code,supplier_id,lead_time_mean,lead_time_variance,disruption_probability,compliance_eligibility,logistics_reliability,product,probability_of_defect,bulk_discount
0,ARE,SUP_ARE_1,29.081672,2.427785,0.283742,0.728946,0.844884,IC Components,0.038476,0.092886
1,AUS,SUP_AUS_2,53.948194,2.954602,0.200381,0.789128,0.779876,Microprocessors,0.075537,0.026052
2,AUS,SUP_AUS_3,53.655471,8.445529,0.226886,0.820285,0.768079,IC Components,0.034397,0.104360
3,BEL,SUP_BEL_4,30.632043,5.234186,0.274002,0.725275,0.795612,Transistors,0.024267,0.085182
4,BEL,SUP_BEL_5,33.266109,2.126099,0.242300,0.801282,0.832547,Power Devices,0.048048,0.048929
...,...,...,...,...,...,...,...,...,...,...
86,THA,SUP_THA_87,36.172719,4.173442,0.415005,0.475663,0.719665,IC Components,0.032679,0.116746
87,THA,SUP_THA_88,36.522238,4.386632,0.424040,0.503805,0.673557,IC Components,0.029301,0.114498
88,THA,SUP_THA_89,39.347376,2.127328,0.471242,0.455950,0.704292,Transistors,0.024677,0.076167
89,USA,SUP_USA_90,11.424375,1.132189,0.315940,0.713011,0.808574,Microprocessors,0.061081,0.040592


# Adding a bulk-units column i.e., bulk discounts only applied when tied to a minimum order quantity.

Depends on:
 - product category
 - country's manufacturing maturity
 - country's cost structure
 - country's alignment with that product (weight matrix)

Country alignment criteria:
 - If a country is strong in a product, MOQ is lower.
 - If weak, MOQ is higher.

In [128]:
# ------------------------------------------------------------
# Base MOQ by product (industry realistic)
# ------------------------------------------------------------
random_state = 42

base_moq = {
    "Microprocessors": 1000,
    "Power Devices": 2000,
    "Transistors": 5000,
    "IC Components": 8000
}

# ------------------------------------------------------------
# Compute MOQ (bulk_units) based on country-product alignment
# ------------------------------------------------------------
def compute_bulk_units(country_code, product):
    weight = country_product_weights[country_code][product]
    base = base_moq[product]

    # Lower weight → higher MOQ; higher weight → lower MOQ
    multiplier = 1.2 - weight   # ranges ~0.7 to ~1.15

    moq = int(base * multiplier) * np.random.uniform(0.9, 1.1)

    # Ensure MOQ is at least 1
    return max(moq, 1)

# ------------------------------------------------------------
# Apply to suppliers DataFrame
# ------------------------------------------------------------
suppliers["bulk_units"] = suppliers.apply(
    lambda row: compute_bulk_units(row["country_code"], row["product"]),
    axis=1
)


In [129]:
suppliers

,country_code,supplier_id,lead_time_mean,lead_time_variance,disruption_probability,compliance_eligibility,logistics_reliability,product,probability_of_defect,bulk_discount,bulk_units
0,ARE,SUP_ARE_1,29.081672,2.427785,0.283742,0.728946,0.844884,IC Components,0.038476,0.092886,6892.535116
1,AUS,SUP_AUS_2,53.948194,2.954602,0.200381,0.789128,0.779876,Microprocessors,0.075537,0.026052,1139.510193
2,AUS,SUP_AUS_3,53.655471,8.445529,0.226886,0.820285,0.768079,IC Components,0.034397,0.104360,7775.663406
3,BEL,SUP_BEL_4,30.632043,5.234186,0.274002,0.725275,0.795612,Transistors,0.024267,0.085182,3714.485041
4,BEL,SUP_BEL_5,33.266109,2.126099,0.242300,0.801282,0.832547,Power Devices,0.048048,0.048929,2035.962102
...,...,...,...,...,...,...,...,...,...,...,...
86,THA,SUP_THA_87,36.172719,4.173442,0.415005,0.475663,0.719665,IC Components,0.032679,0.116746,5920.365381
87,THA,SUP_THA_88,36.522238,4.386632,0.424040,0.503805,0.673557,IC Components,0.029301,0.114498,5619.146972
88,THA,SUP_THA_89,39.347376,2.127328,0.471242,0.455950,0.704292,Transistors,0.024677,0.076167,4601.576616
89,USA,SUP_USA_90,11.424375,1.132189,0.315940,0.713011,0.808574,Microprocessors,0.061081,0.040592,627.869936


In [130]:
products

,date,ppi_value,country_code,ppi_source,real_price,product,ppi_region,year,month
0,1976-01-01,100.0,ARE,FRED,0.260000,integrated_circuit_components,OAC,1976,1
1,1976-02-01,100.0,ARE,FRED,0.260000,integrated_circuit_components,OAC,1976,2
2,1976-03-01,100.0,ARE,FRED,0.260000,integrated_circuit_components,OAC,1976,3
3,1976-04-01,100.0,ARE,FRED,0.260000,integrated_circuit_components,OAC,1976,4
4,1976-05-01,100.0,ARE,FRED,0.260000,integrated_circuit_components,OAC,1976,5
...,...,...,...,...,...,...,...,...,...
36125,2025-12-01,98.7,BRA,FRED,0.041454,transistors,OAC,2025,12
36126,2025-12-01,98.7,IND,FRED,0.031584,transistors,OAC,2025,12
36127,2025-12-01,98.7,THA,FRED,0.034545,transistors,OAC,2025,12
36128,2025-12-01,98.7,IDN,FRED,0.032571,transistors,OAC,2025,12


# Creating a baseline price column

In [131]:
product_name_map = {
    "IC Components": "integrated_circuit_components",
    "Transistors": "transistors",
    "Power Devices": "power_devices",
    "Microprocessors": "microprocessors"
}


In [132]:
products.loc[lambda x: x['product'] == 'transistors']

,date,ppi_value,country_code,ppi_source,real_price,product,ppi_region,year,month
24370,1977-01-01,109.0,USA,BLS,0.070964,transistors,USA,1977,1
24371,1977-01-01,NaN,JPN,FRED,NaN,transistors,OAC,1977,1
24372,1977-01-01,NaN,DEU,FRED,NaN,transistors,USA,1977,1
24373,1977-01-01,NaN,FRA,FRED,NaN,transistors,USA,1977,1
24374,1977-01-01,NaN,GBR,FRED,NaN,transistors,USA,1977,1
...,...,...,...,...,...,...,...,...,...
36125,2025-12-01,98.7,BRA,FRED,0.041454,transistors,OAC,2025,12
36126,2025-12-01,98.7,IND,FRED,0.031584,transistors,OAC,2025,12
36127,2025-12-01,98.7,THA,FRED,0.034545,transistors,OAC,2025,12
36128,2025-12-01,98.7,IDN,FRED,0.032571,transistors,OAC,2025,12


In [133]:
# Make a copy of products with normalized product names
products_copy = products.copy()
products_copy["product_norm"] = products_copy["product"].str.lower().str.strip()

# Compute most recent real_price per (country_code, product)
latest_prices = (
    products_copy.sort_values("date")
    .groupby(["country_code", "product_norm"])["real_price"]
    .last()
    .reset_index()
)


In [134]:
# Normalize supplier product names to match products table
suppliers["product_norm"] = suppliers["product"].map(product_name_map)

# Merge baseline prices
suppliers = suppliers.merge(
    latest_prices,
    how="left",
    left_on=["country_code", "product_norm"],
    right_on=["country_code", "product_norm"]
)

# Rename for clarity
suppliers = suppliers.rename(columns={"real_price": "baseline_price"})


In [135]:
suppliers = suppliers.drop(columns=["product_norm"])


In [140]:
suppliers.drop(suppliers.loc[lambda x: x['baseline_price'].isna()].index, axis=0, inplace=True)

In [142]:
suppliers.loc[lambda x: x['country_code']== 'THA']

,country_code,supplier_id,lead_time_mean,lead_time_variance,disruption_probability,compliance_eligibility,logistics_reliability,product,probability_of_defect,bulk_discount,bulk_units,baseline_price
79,THA,SUP_THA_80,41.109620,7.588064,0.480488,0.489290,0.689975,Power Devices,0.041260,0.054374,1732.640675,0.076986
80,THA,SUP_THA_81,39.229602,6.467865,0.460113,0.504247,0.691064,IC Components,0.030616,0.104744,5797.205325,0.140859
81,THA,SUP_THA_82,36.054460,3.272204,0.441641,0.495730,0.681433,IC Components,0.028478,0.118037,5087.337173,0.140859
82,THA,SUP_THA_83,39.555703,6.169221,0.418231,0.482693,0.710013,Transistors,0.025020,0.072641,4713.813699,0.034545
83,THA,SUP_THA_84,36.868560,4.166941,0.495257,0.500050,0.720823,Power Devices,0.041068,0.050930,1952.957563,0.076986
84,THA,SUP_THA_85,42.870928,4.589640,0.412885,0.477200,0.724921,IC Components,0.032730,0.123133,5662.787046,0.140859
85,THA,SUP_THA_86,44.277929,4.996906,0.480842,0.496318,0.714300,IC Components,0.028033,0.112117,5642.763144,0.140859
86,THA,SUP_THA_87,36.172719,4.173442,0.415005,0.475663,0.719665,IC Components,0.032679,0.116746,5920.365381,0.140859
87,THA,SUP_THA_88,36.522238,4.386632,0.424040,0.503805,0.673557,IC Components,0.029301,0.114498,5619.146972,0.140859
88,THA,SUP_THA_89,39.347376,2.127328,0.471242,0.455950,0.704292,Transistors,0.024677,0.076167,4601.576616,0.034545


In [144]:
suppliers = suppliers.assign(
    bulk_units = lambda x: x.bulk_units.astype(int)
)

In [147]:
suppliers.tail()

,country_code,supplier_id,lead_time_mean,lead_time_variance,disruption_probability,compliance_eligibility,logistics_reliability,product,probability_of_defect,bulk_discount,bulk_units,baseline_price
86,THA,SUP_THA_87,36.172719,4.173442,0.415005,0.475663,0.719665,IC Components,0.032679,0.116746,5920,0.140859
87,THA,SUP_THA_88,36.522238,4.386632,0.424040,0.503805,0.673557,IC Components,0.029301,0.114498,5619,0.140859
88,THA,SUP_THA_89,39.347376,2.127328,0.471242,0.455950,0.704292,Transistors,0.024677,0.076167,4601,0.034545
89,USA,SUP_USA_90,11.424375,1.132189,0.315940,0.713011,0.808574,Microprocessors,0.061081,0.040592,627,0.315585
90,USA,SUP_USA_91,11.467816,2.209626,0.313827,0.786326,0.776687,Microprocessors,0.052848,0.039968,568,0.315585


# Injecting Some Noise into Baseline Prices by Supplier

In [146]:
random_state = 42

# Product-specific noise ranges (multiplicative)
product_noise_ranges = {
    "Microprocessors": 0.03,   # ±3%
    "Power Devices": 0.05,     # ±5%
    "Transistors": 0.08,       # ±8%
    "IC Components": 0.10      # ±10%
}

def add_product_specific_noise(row):
    base_price = row["baseline_price"]
    product = row["product"]

    # Get noise scale for this product
    noise_scale = product_noise_ranges[product]

    # Generate multiplicative noise factor
    noise_factor = np.random.uniform(1 - noise_scale, 1 + noise_scale)

    return base_price * noise_factor




In [148]:
# Apply to suppliers DataFrame
suppliers["baseline_price"] = suppliers.apply(add_product_specific_noise, axis=1)

In [149]:
suppliers.tail()

,country_code,supplier_id,lead_time_mean,lead_time_variance,disruption_probability,compliance_eligibility,logistics_reliability,product,probability_of_defect,bulk_discount,bulk_units,baseline_price
86,THA,SUP_THA_87,36.172719,4.173442,0.415005,0.475663,0.719665,IC Components,0.032679,0.116746,5920,0.130497
87,THA,SUP_THA_88,36.522238,4.386632,0.424040,0.503805,0.673557,IC Components,0.029301,0.114498,5619,0.153702
88,THA,SUP_THA_89,39.347376,2.127328,0.471242,0.455950,0.704292,Transistors,0.024677,0.076167,4601,0.035908
89,USA,SUP_USA_90,11.424375,1.132189,0.315940,0.713011,0.808574,Microprocessors,0.061081,0.040592,627,0.316023
90,USA,SUP_USA_91,11.467816,2.209626,0.313827,0.786326,0.776687,Microprocessors,0.052848,0.039968,568,0.319637


In [150]:
products

,date,ppi_value,country_code,ppi_source,real_price,product,ppi_region,year,month
0,1976-01-01,100.0,ARE,FRED,0.260000,integrated_circuit_components,OAC,1976,1
1,1976-02-01,100.0,ARE,FRED,0.260000,integrated_circuit_components,OAC,1976,2
2,1976-03-01,100.0,ARE,FRED,0.260000,integrated_circuit_components,OAC,1976,3
3,1976-04-01,100.0,ARE,FRED,0.260000,integrated_circuit_components,OAC,1976,4
4,1976-05-01,100.0,ARE,FRED,0.260000,integrated_circuit_components,OAC,1976,5
...,...,...,...,...,...,...,...,...,...
36125,2025-12-01,98.7,BRA,FRED,0.041454,transistors,OAC,2025,12
36126,2025-12-01,98.7,IND,FRED,0.031584,transistors,OAC,2025,12
36127,2025-12-01,98.7,THA,FRED,0.034545,transistors,OAC,2025,12
36128,2025-12-01,98.7,IDN,FRED,0.032571,transistors,OAC,2025,12


# Creating a Price Volatility for each supplier based on their disruption probability and rolling standard deviation of real_price of product (grouped by country in products table) over last 5 years

In [158]:
product_map = {
    "IC Components": "integrated_circuit_components",
    "Transistors": "transistors",
    "Power Devices": "power_devices",
    "Microprocessors": "microprocessors"
}

suppliers["product_norm"] = suppliers["product"].map(product_map)
products["product_norm"] = products["product"].str.lower().str.strip()


In [159]:
products_sorted = products.sort_values(["country_code", "product_norm", "date"])

products_sorted["price_volatility_raw"] = (
    products_sorted
    .groupby(["country_code", "product_norm"])["real_price"]
    .rolling(60, min_periods=12)
    .std()
    .reset_index(level=[0,1], drop=True)
)


In [160]:
latest_volatility = (
    products_sorted
    .groupby(["country_code", "product_norm"])["price_volatility_raw"]
    .last()
    .reset_index()
)


In [161]:
suppliers = suppliers.merge(
    latest_volatility,
    how="left",
    left_on=["country_code", "product_norm"],
    right_on=["country_code", "product_norm"]
)


In [162]:
min_v = suppliers["price_volatility_raw"].min()
max_v = suppliers["price_volatility_raw"].max()

suppliers["price_volatility_norm"] = (
    (suppliers["price_volatility_raw"] - min_v) / (max_v - min_v)
)


In [163]:
suppliers["price_volatility"] = (
    0.6 * suppliers["price_volatility_norm"] +
    0.4 * suppliers["disruption_probability"]
)


In [164]:
suppliers = suppliers.drop(columns=["price_volatility_raw", "price_volatility_norm",
                                    'product_norm'])


In [165]:
suppliers

,country_code,supplier_id,lead_time_mean,lead_time_variance,disruption_probability,compliance_eligibility,logistics_reliability,product,probability_of_defect,bulk_discount,bulk_units,baseline_price,price_volatility
0,ARE,SUP_ARE_1,29.081672,2.427785,0.283742,0.728946,0.844884,IC Components,0.038476,0.092886,6892,0.246855,0.172129
1,AUS,SUP_AUS_2,53.948194,2.954602,0.200381,0.789128,0.779876,Microprocessors,0.075537,0.026052,1139,0.991371,0.680152
2,AUS,SUP_AUS_3,53.655471,8.445529,0.226886,0.820285,0.768079,IC Components,0.034397,0.104360,7775,0.213238,0.144877
3,BEL,SUP_BEL_4,30.632043,5.234186,0.274002,0.725275,0.795612,Transistors,0.024267,0.085182,3714,0.044963,0.111905
4,BRA,SUP_BRA_6,40.588272,7.953977,0.482714,0.440989,0.678642,Transistors,0.022972,0.087375,3455,0.040343,0.201362
...,...,...,...,...,...,...,...,...,...,...,...,...,...
83,THA,SUP_THA_87,36.172719,4.173442,0.415005,0.475663,0.719665,IC Components,0.032679,0.116746,5920,0.130497,0.202083
84,THA,SUP_THA_88,36.522238,4.386632,0.424040,0.503805,0.673557,IC Components,0.029301,0.114498,5619,0.153702,0.205698
85,THA,SUP_THA_89,39.347376,2.127328,0.471242,0.455950,0.704292,Transistors,0.024677,0.076167,4601,0.035908,0.195393
86,USA,SUP_USA_90,11.424375,1.132189,0.315940,0.713011,0.808574,Microprocessors,0.061081,0.040592,627,0.316023,0.341034


In [169]:
suppliers.loc[lambda x: x['product'] == 'IC Components']

,country_code,supplier_id,lead_time_mean,lead_time_variance,disruption_probability,compliance_eligibility,logistics_reliability,product,probability_of_defect,bulk_discount,bulk_units,baseline_price,price_volatility
0,ARE,SUP_ARE_1,29.081672,2.427785,0.283742,0.728946,0.844884,IC Components,0.038476,0.092886,6892,0.246855,0.172129
2,AUS,SUP_AUS_3,53.655471,8.445529,0.226886,0.820285,0.768079,IC Components,0.034397,0.104360,7775,0.213238,0.144877
5,CAN,SUP_CAN_7,9.938467,0.783581,0.192255,0.839325,0.853692,IC Components,0.031026,0.107543,5892,0.214622,0.077053
8,CHN,SUP_CHN_11,32.160863,5.674070,0.442243,0.464401,0.716814,IC Components,0.027998,0.109955,6499,0.115502,0.176897
9,CHN,SUP_CHN_12,30.084516,2.712454,0.394882,0.534503,0.780303,IC Components,0.028072,0.104493,6570,0.121953,0.157953
10,CHN,SUP_CHN_13,30.080156,4.461362,0.410360,0.531718,0.748906,IC Components,0.030958,0.098670,5669,0.127232,0.164144
11,CHN,SUP_CHN_14,29.800068,2.592431,0.424252,0.467710,0.758361,IC Components,0.031102,0.104478,5774,0.120585,0.169701
12,CHN,SUP_CHN_15,33.360055,3.666606,0.395581,0.539868,0.752716,IC Components,0.028678,0.109368,5476,0.127156,0.158232
15,CHN,SUP_CHN_18,33.772080,6.049152,0.378160,0.512684,0.765665,IC Components,0.030137,0.115140,6511,0.115446,0.151264
16,DEU,SUP_DEU_19,39.284839,3.968803,0.215641,0.747076,0.790670,IC Components,0.034863,0.093799,8617,0.216882,0.097436


# Considering How to Add Tariff Info

In [166]:
tariff = pd.read_csv('/content/tarriff_database_2025_only_semiconductor_components_v2.csv')

In [168]:
tariff.loc[lambda x: x.mfn_text_rate_pct > 0]

,hts8,brief_description,mfn_text_rate_pct,how_measured
10,84807180,"Molds for rubber or plastics, injection or com...",3.1,NO
14,84879000,"Machinery parts, not containing electrical con...",3.9,KG
17,85042300,Liquid dielectric transformers having a power ...,1.6,NO
19,85043140,Electrical transformers other than liquid diel...,6.6,NO
20,85043160,Electrical transformers other than liquid diel...,1.6,NO
21,85043200,Electrical transformers other than liquid diel...,2.4,NO
22,85043300,Electrical transformers other than liquid diel...,1.6,NO
23,85043400,Electrical transformers other than liquid diel...,1.6,NO
31,85118020,Voltage and voltage-current regulators with cu...,2.5,NO
33,85119020,Parts of voltage and voltage-current regulator...,3.1,KG


Code 85389030 can potentially be included under IC components

In [171]:
suppliers = suppliers.assign(
    hts8_tariff = lambda x: np.where(x['product'] == 'IC Components', '85389030', 'None')
)

In [172]:
suppliers.to_csv('suppliers_products.csv', index=False)